<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hemekci/AIE640/blob/main/Lab-Notebooks/Chapter_1_Introduction_to_Language_Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

## Understanding What We're Building

As discussed in Chapter 1, Large Language Models are built on the transformer architecture from the 2017 "Attention is All You Need" paper. The model we're using today, **Phi-3**, is a **decoder-only model** - similar to GPT. These models generate text autoregressively, meaning they predict one token at a time based on all previous tokens.

Remember from Chapter 1: the "4k" in Phi-3-mini-4k-instruct refers to the context window - it can process up to 4,000 tokens at once. This is important because, as we learned, the context length determines how much text the model can "remember" during generation.

### Generating Your First Text

The main source for finding and downloading LLMs is the [HuggingFace Hub](https://huggingface.co/docs/hub/en/index)

**HuggingFace** is the organization behind the well-known Transformers
package, discussed heavily in the slides.

In [ ]:
%%capture
!pip install transformers>=4.40.1 accelerate>=0.27.2

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Although tokenization will be discussed more comprehensively in Chapter 2, let's have a quick look of how it works:

In [5]:
# Understanding Tokenization from Chapter 1
# We know from Chapter 1 that tokenization is converting text to numbers

# Let's see how Phi-3 tokenizes text
sample_text = "I love llamas"
tokens = tokenizer(sample_text)
print(f"Text: {sample_text}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Back to text: {tokenizer.decode(tokens['input_ids'])}")

# Notice how the tokenizer splits text into subword units
# This is more sophisticated than the simple word-based tokenization we saw in Chapter 1

Text: I love llamas
Token IDs: [306, 5360, 11829, 294]
Back to text: I love llamas


Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [6]:
from transformers import pipeline

# Create a pipeline with hyperparameters we learned about
# As discussed in Chapter 1, these control how the model generates text

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # Only return newly generated text
    max_new_tokens=500,      # Maximum tokens to generate (remember context windows!)
    do_sample=False          # Deterministic output (greedy decoding)
)

"""
Key generation parameters from Chapter 1:

- max_new_tokens: Limits generation length. Remember that GPT-style models
  are autoregressive - they generate one token at a time.

- do_sample: When False, always picks the most likely next token (greedy).
  When True, samples from the probability distribution.

- temperature (not set here): Controls randomness. Low values make the model
  more focused and deterministic, high values make it more creative.

- top_p and top_k: Control nucleus and top-k sampling as alternatives to
  temperature-based sampling.
"""

'\nKey generation parameters from Chapter 1:\n\n- max_new_tokens: Limits generation length. Remember that GPT-style models\n  are autoregressive - they generate one token at a time.\n\n- do_sample: When False, always picks the most likely next token (greedy).\n  When True, samples from the probability distribution.\n\n- temperature (not set here): Controls randomness. Low values make the model\n  more focused and deterministic, high values make it more creative.\n\n- top_p and top_k: Control nucleus and top-k sampling as alternatives to\n  temperature-based sampling.\n'

Finally, we create our prompt as a user and give it to the model:

In [7]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

Why did the chicken join the band? Because it had the drumsticks!


### Your Turn: Applying Chapter 1 Concepts

Now let's practice the concepts from Chapter 1 with hands-on exercises.


#### Exercise 1: Token Vocabulary Exploration
Run this to see how different types of text tokenize differently. Try using different characters.

In [8]:
# Exercise 1: Now modify test_texts and add your own examples
print("Different text types tokenize differently:")
print("=" * 60)

test_texts = {
    "English": "The cat sat on the mat",
    "Code": "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "Numbers": "3.14159 2.71828 1.41421",
    "Mixed": "GPT-3 has 175B parameters.",
    "Special": "Hello 世界 🦙 #AI"
}

for text_type, text in test_texts.items():
    tokens = tokenizer(text)
    print(f"\n{text_type}:")
    print(f"  Text: '{text[:50]}{'...' if len(text) > 50 else ''}'")
    print(f"  Tokens: {len(tokens['input_ids'])}")
    print(f"  First 5 token IDs: {tokens['input_ids'][:5]}")


Different text types tokenize differently:

English:
  Text: 'The cat sat on the mat'
  Tokens: 6
  First 5 token IDs: [450, 6635, 3290, 373, 278]

Code:
  Text: 'def fibonacci(n): return n if n <= 1 else fibonacc...'
  Tokens: 32
  First 5 token IDs: [822, 18755, 265, 21566, 29898]

Numbers:
  Text: '3.14159 2.71828 1.41421'
  Tokens: 24
  First 5 token IDs: [29871, 29941, 29889, 29896, 29946]

Mixed:
  Text: 'GPT-3 has 175B parameters.'
  Tokens: 12
  First 5 token IDs: [402, 7982, 29899, 29941, 756]

Special:
  Text: 'Hello 世界 🦙 #AI'
  Tokens: 11
  First 5 token IDs: [15043, 29871, 30793, 30967, 29871]


#### Questions:

**Q1)** Why does the code example tokenize differently from plain English text (32 tokens vs 6 tokens)? What does this tell you about how the tokenizer handles programming syntax versus natural language?

In coding most symbols such as = are its own token. It comes down to what keeps repeating in the training data. So it means that tokenizer was not trained for programming syntax. It probably got trained mostly for English and that's why it has less tokens.

**Q2)** How would tokenization differ if you input text in a non-English language like Chinese or Arabic? What effects does this have for model performance across different languages?

First important thing is again training data. If it got train mostly for English, then it is going to look for tokens that are mostly used in English. Second factor may be  how a language works. For example Turkish is an agglutinative language so it is only logical for it to have lots of tokens after the verbs for example.

So, even though it is the same content, it takes more tokens for a sentence that is in models less trained language. And more tokens mean more expensive

**Q3)** Why do numbers like "3.14159" result in many more tokens (24 total) than you might expect? What does this tell about how transformers process numerical data?

It is because the tokenizer treats every digit and the decimal point as separate tokens and puts a space token before each number This is a design choice. Every number is split the same way, the model can keep track of digit positions more easily, which helps with arithmetic. It also shows that models do not see numbers as values but as symbols. They have to learn numerical relationships from patterns in these digit tokens.

**Q4)** What would happen if you tried to process text with a vocabulary that wasn't seen during the tokenizer's training? How might this affect the model's ability to understand and generate responses?

It differs but accoring to my research, byte based tokenizers splits words until all the tokens are known to it so It understands but tokenizers like BERT mark it as UNK. Also if the parts that are tokenizd are meaningful, model gets the word through different words in its training data. But if its completely new it struggles to understand.

**Q5)** Why is it important that the tokenizer can handle special characters and mixed content (like "GPT-3 has 175B parameters")? How does subword tokenization help with out-of-vocabulary words?

It is because inputs may vary. Lots of naturel and coding languages have different symbols and words. Model should be able to understand them.

As I explained in the previous question if a word is unknown, it splits the word into smaller pieces and try to make it meaningful.

#### Exercise 2: Autoregressive Generation Steps
Watch how the model generates text token by token:

In [14]:
# Exercise 2: Change the prompt to see different patterns
print("AUTOREGRESSIVE GENERATION")
print("Chapter 1: Decoder models generate one token at a time")
print("=" * 60)

prompt = [{"role": "user", "content": "The movie twelve angry man"}]

# Generate different lengths to see the progression
for num_tokens in [5, 15, 30]:
    output = generator(prompt, max_new_tokens=num_tokens, do_sample=False)
    print(f"\nAfter {num_tokens} tokens:")
    print(f"'{output[0]['generated_text']}'")

print("\nNotice how each output builds on the previous tokens!")



AUTOREGRESSIVE GENERATION
Chapter 1: Decoder models generate one token at a time

After 5 tokens:
'I'm sorry,'

After 15 tokens:
'I'm sorry, but it seems like you're referring to a'

After 30 tokens:
'I'm sorry, but it seems like you're referring to a movie that doesn't exist or isn't widely recognized. It''

Notice how each output builds on the previous tokens!


#### Questions:

**Q1)** Why does the model generate text one token at a time rather than producing the entire output at once? What are the computational implications of this approach?

Model checks the previous tokens and chooses the next token. Than places that token after the last produced token. Than cheks all the previous tokens and then create another one. Each token is connected to the tokens before it.

**Q2)** How does the context window limitation (4,000 tokens for Phi-3) affect what the model can "remember" during generation? What happens when you exceed this limit?

Model remembers my prompt and the tokens that are previously produced. It only remembers those informations. If the limit is exceeded then it may stop producing or starts forgetting.

**Q3)** What patterns do you notice in how the model completes the sentence as more tokens are generated? Why does the output become more coherent with additional tokens?

Each longer output starts with exactly the same text as the shorter one, because the model only adds new tokens and never changes the previous ones. It is because model produces tokens regarding all of the previous tokens produced. That's why it looks more coherent.

**Q4)** Why might setting `do_sample=False` (greedy decoding) produce more predictable but potentially less creative outputs? When would you want deterministic vs. probabilistic generation?

Because its output is only deterministic. It only prints the most probablistic result thus making it predictable and always the same. I would want more deterministic answers while dealing with "only one true answer" kind of stuff. I would want more probabilistic answers while trying to write a text, mail or needing some kind of creativity.


**Q5)** How would changing the prompt from "The attention mechanism in transformers" to a more specific technical question affect the autoregressive generation process and the quality of intermediate outputs?

I asked about a movie that is called 12 angry man and it did not know it. So it gave the output "I'm sorry, but it seems like you're referring to a movie that doesn't exist or isn't widely recognized." Model did not know the movie at all so it said that it does nor exist.

#### Challenge 3: Comparing Models - Decoder Architecture
Chapter 1 discussed how decoder-only models like GPT generate text.

Load Qwen/Qwen2.5-1.5B-Instruct and compare it (This could take a little long):

In [15]:
# Exercise 3: Comparing decoder-only models
print("LOADING QWEN MODEL FOR COMPARISON")
print("=" * 60)

# Load Qwen (smaller model for faster loading)
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("Loading Qwen/Qwen2.5-1.5B-Instruct...")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="cuda",
    torch_dtype="auto"
)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

qwen_generator = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False
)

# Compare outputs for different prompts
test_prompts = [
    "Explain neural networks in simple terms:",
    "Write a Python function to sort a list:",
    "What is the meaning of life?"
]

for prompt_text in test_prompts:
    prompt = [{"role": "user", "content": prompt_text}]

    print(f"\nPrompt: '{prompt_text}'")
    print("-" * 40)

    phi3_output = generator(prompt, max_new_tokens=30, do_sample=False)
    print(f"Phi-3: {phi3_output[0]['generated_text'][:100]}...")

    qwen_output = qwen_generator(prompt, max_new_tokens=30, do_sample=False)
    print(f"Qwen:  {qwen_output[0]['generated_text'][:100]}...")

print("\nNotice the different styles and capabilities of each model!")

LOADING QWEN MODEL FOR COMPARISON
Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


Prompt: 'Explain neural networks in simple terms:'
----------------------------------------
Phi-3: Neural networks are a series of algorithms that attempt to recognize underlying relationships in a s...
Qwen:  Neural networks are a type of machine learning model inspired by the structure and function of the h...

Prompt: 'Write a Python function to sort a list:'
----------------------------------------
Phi-3: Certainly! Below is a Python function that sorts a list using the built-in `sorted()` function, whic...
Qwen:  Certainly! Here's an example of a Python function that sorts a list using the built-in `sorted()` fu...

Prompt: 'What is the meaning of life?'
----------------------------------------
Phi-3: The meaning of life is a philosophical question concerning the significance of life or existence in ...
Qwen:  As an AI language model, I don't have personal beliefs or opinions about what constitutes the "meani...

Notice the different styles and capabilities of each model!


#### Questions:

**Q1)** Why do Phi-3 (7.6B parameters) and Qwen (1.5B parameters) produce different styles of responses to the same prompt? How does model size affect response quality and style?

The data sets that they are trained are different. It is expected for them to give adifferent outputs. Model size means it is trained with more data. So I would expect the answers to be better. But from those outputs I don't think I could make a good comment about quality.

**Q2)** What architectural differences between these decoder-only models might account for their varying approaches to explaining concepts or generating code?

I don't actually know what would be the architectural difference.

**Q3)** How does the training data for each model influence their responses? Why might one model be better at technical explanations while another excels at creative tasks?

Their temperatures are probably different. One is more deterministic while other is more creative.

**Q4)** Why do both models use the same fundamental autoregressive generation approach despite their size differences? What does this tell you about the transformer architecture's scalability?

Both are decoder-only transformers trained with the same objective so generation is the same loop for both. This shows the transformer scales well. The same building blocks  can simply be stacked wider and deeper, from 1.5B to hundreds of billions of parameters, without changing the design.

**Q5)** What trade-offs are involved in choosing a smaller model like Qwen versus a larger one like Phi-3? Consider factors like inference speed, memory usage, and task performance.

A smaller model like Qwen is faster, because each token needs fewer computations, and it uses less memory. so it runs on cheaper hardware. The trade-off is task performance. The right choice depends on the task.

#### Challenge 4: Autoregressive Generation
Chapter 1 explained that decoder models generate text one token at a time. Let's observe this:

In [16]:
# Exercise 4: Try different prompts, temperatures, top_p values, etc.
print("EXPERIMENT")
print("=" * 60)
print("Change these examples and play with the models:")

# Example 1: Story generation with different temperatures
story_prompt = [{"role": "user", "content": "Once upon a time in a world where"}]

print("\n1. Story Generation:")
for temp in [0.5, 1.0, 1.5, 5.0]:
    output = generator(story_prompt, max_new_tokens=40, do_sample=True, temperature=temp)
    print(f"\nTemperature {temp}: {output[0]['generated_text'][:80]}...")

# Example 2: Technical explanation
tech_prompt = [{"role": "user", "content": "The transformer architecture works by"}]

print("\n\n2. Technical Explanation:")
output = generator(tech_prompt, max_new_tokens=50, do_sample=False)
print(output[0]['generated_text'])


EXPERIMENT
Change these examples and play with the models:

1. Story Generation:

Temperature 0.5: Once upon a time in a world where the sun never set on the sprawling city of Neo...

Temperature 1.0: Once upon a time in a world where magic was as common as the air we breathe, and...

Temperature 1.5: The moment our story unfurls is as vast and as variable as the limitless clouds,...

Temperature 5.0: Une, mon apOintement peut avoir comme base soit DIOYNE, à savolIeor LunHorm ± es...


2. Technical Explanation:
The transformer architecture works by using self-attention mechanisms to process input data in parallel, rather than sequentially. This allows for more efficient handling of long-range dependencies in data, such as in natural language processing tasks. The architecture consists


In [ ]:
# Experiment with the hyperparameters:
'''
print("\nExperiment:")
temp2 = x.x # change to a float value
output = generator(story_prompt, max_new_tokens=40, do_sample=True, temperature=temp2)
print(f"\nTemperature {temp2}: {output[0]['generated_text'][:80]}...")
'''

'\nprint("\nExperiment:")\ntemp2 = x.x # change to a float value\noutput = generator(story_prompt, max_new_tokens=40, do_sample=True, temperature=temp2)\nprint(f"\nTemperature {temp2}: {output[0][\'generated_text\'][:80]}...")\n'

#### Questions:

**Q1)** Why does temperature 0.5 produce more coherent but predictable text while temperature 5.0 generates seemingly random or nonsensical output? What is happening to the probability distribution at different temperature values?

---

**Q2)** How does temperature mathematically modify the softmax distribution over the vocabulary? Why does dividing logits by temperature before softmax affect randomness?

---

**Q3)** What types of tasks would benefit from low temperature (0.5) versus high temperature (1.5) settings? Consider use cases like technical documentation vs. creative writing.

---

**Q4)** Why does the model sometimes produce grammatically incorrect or illogical text at very high temperatures even though it was trained on coherent text? What does this reveal about how language models store and retrieve information?

---

**Q5)** How would combining temperature with other sampling strategies like top-p (nucleus sampling) or top-k affect the generation quality? Why might you want to use multiple sampling techniques together?